# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 1. Load the dataset
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url)

# Display initial shape and columns
print(f"Initial shape: {df.shape}")

# 2. Define features and target variable
target_column = 'is_canceled'
X_raw = df.drop(columns=[target_column])
y = df[target_column]

# 3. Train-test split (70/30) BEFORE preprocessing/transformations
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.30, random_state=42, stratify=y
)

# 4. Identify column types
# Drop highly unique, uninformative, or leaking identifiers/dates to prevent overfitting
cols_to_drop = ['reservation_status', 'reservation_status_date', 'country', 'agent', 'company']
X_train_raw = X_train_raw.drop(columns=[col for col in cols_to_drop if col in X_train_raw.columns])
X_test_raw = X_test_raw.drop(columns=[col for col in cols_to_drop if col in X_test_raw.columns])

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()

# 5. Impute missing values
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# Fit on training data and transform both sets
X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train_raw[num_cols]), columns=num_cols, index=X_train_raw.index)
X_test_num = pd.DataFrame(num_imputer.transform(X_test_raw[num_cols]), columns=num_cols, index=X_test_raw.index)

X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train_raw[cat_cols]), columns=cat_cols, index=X_train_raw.index)
X_test_cat = pd.DataFrame(cat_imputer.transform(X_test_raw[cat_cols]), columns=cat_cols, index=X_test_raw.index)

# 6. Encode categorical variables using OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit on training categorical data and transform
X_train_encoded_array = encoder.fit_transform(X_train_cat)
X_test_encoded_array = encoder.transform(X_test_cat)

encoded_cols = encoder.get_feature_names_out(cat_cols)
X_train_encoded = pd.DataFrame(X_train_encoded_array, columns=encoded_cols, index=X_train_raw.index)
X_test_encoded = pd.DataFrame(X_test_encoded_array, columns=encoded_cols, index=X_test_raw.index)

# 7. Reconstruct final X_train and X_test
X_train = pd.concat([X_train_num, X_train_encoded], axis=1)
X_test = pd.concat([X_test_num, X_test_encoded], axis=1)

# Verify final shapes
print(f"X_train shape after processing: {X_train.shape}")
print(f"X_test shape after processing: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Initial shape: (9404, 32)
X_train shape after processing: (6582, 70)
X_test shape after processing: (2822, 70)
y_train shape: (6582,)
y_test shape: (2822,)


### ✍️ Your Response: 🔧
1. The intial shape was 9404 rows and 32 columns.

2. Categorical included classifications like where the hotel was located (resort/city), room resrve type and customer type. Numerical were things like the number of guest (adults/children), lead time and stay time.

3. After doing the training testing split some variables were removed to avoid leakage such as reservation status. Missing vairables were added using median imputation for numeric and mode for categorical.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Naïve Bayes model
nb_model = GaussianNB()

# Train the model
nb_model.fit(X_train, y_train)

# Predict on the test set
y_pred = nb_model.predict(X_test)

# Output performance metrics
print("--- Naïve Bayes Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Naïve Bayes Classification Report ---
              precision    recall  f1-score   support

           0       0.94      0.56      0.70      2121
           1       0.40      0.89      0.55       701

    accuracy                           0.64      2822
   macro avg       0.67      0.72      0.63      2822
weighted avg       0.80      0.64      0.66      2822

--- Confusion Matrix ---
[[1187  934]
 [  80  621]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. Since accuracy can be a sign of over or under fit, F-1 score is likely going to be a better reflection of the overall output to ensure a relative balance between preciesion and recall (ie mistakes in either direction), here the F-1 score seemed pretty high suggesting that the model was well balanced during training.

2. This model would likely be used as a good predictor for pattern recognition, not necessarily as a real time tool since it would take a lot of time and money to constantly be inputting new data to train on. It could however idientidy patterns and past trends that could be relevant for hotel preparation.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale features using StandardScaler (fit ONLY on X_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train SVM classifier with linear kernel
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 3. Predict on test set
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluate with classification metrics
print("--- SVM (Linear) Classification Report ---")
print(classification_report(y_test, y_pred_svm))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_svm))

--- SVM (Linear) Classification Report ---
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      2121
           1       0.98      0.41      0.58       701

    accuracy                           0.85      2822
   macro avg       0.91      0.70      0.74      2822
weighted avg       0.87      0.85      0.83      2822

--- Confusion Matrix ---
[[2114    7]
 [ 412  289]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. Here again since we are balancing precision and recall (ie roughly equal stakes either way) and avoinging relying solely on the accuracy since that can over or under fit, the F-1 score would be the best metric to judge based off of. Here the F-1 score was actually higher than for naive bayes indicating a better balance of the components within the model.

2. SVM could likely help for more complex modeling used across a hotel chain rather than just a single location, which could help indicate patterns of use or cancellation that would help to guide resource allocation to certain hotel locations during busy seasons and lower costs during slow times.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. Create a true validation split from the training data
X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

# 2. Scale the partitioned sets
scaler_mlp = StandardScaler()
X_sub_train_scaled = scaler_mlp.fit_transform(X_sub_train)
X_val_scaled = scaler_mlp.transform(X_val)
X_test_scaled_mlp = scaler_mlp.transform(X_test)

# 3. Initialize MLPClassifier with 2 hidden layers (e.g., 64 and 32 nodes)
# Use early_stopping and validation_fraction to leverage the validation split natively
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15
)

# 4. Train the model on the training subset
mlp_model.fit(X_sub_train_scaled, y_sub_train)

# 5. Evaluate on the true validation set
val_accuracy = mlp_model.score(X_val_scaled, y_val)
print(f"Validation Set Accuracy: {val_accuracy:.4f}\n")

# 6. Predict and evaluate on the test set
y_pred_mlp = mlp_model.predict(X_test_scaled_mlp)

print("--- Neural Network (MLP) Test Classification Report ---")
print(classification_report(y_test, y_pred_mlp))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_mlp))

Validation Set Accuracy: 0.8745

--- Neural Network (MLP) Test Classification Report ---
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      2121
           1       0.78      0.60      0.68       701

    accuracy                           0.86      2822
   macro avg       0.83      0.77      0.80      2822
weighted avg       0.85      0.86      0.85      2822

--- Confusion Matrix ---
[[2001  120]
 [ 277  424]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. This model actually seemed to be pretty in line with what the SVM model had produced, F-1 scores were similar with only .01 higher accuracy.

2. Considering the current liability landscape and the cost of running a model like this regularly, I don't know that a customer would be comfortable using a model like this regularly. While the black box would mean that the data is only trained within the confined environment of the compnay menaing they control their data the whole way through, from a customer perspective knowledge about these topics is limited at best, and could cause a negative perception of the companies data usage practices merely based off of hearsay in the market. Additionally models like this that can learn as they work are expensive to run as they are constantly generating new iterations.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [5]:
from sklearn.metrics import accuracy_score

# Calculate accuracies on the test set
nb_acc = accuracy_score(y_test, y_pred)
svm_acc = accuracy_score(y_test, y_pred_svm)
mlp_acc = accuracy_score(y_test, y_pred_mlp)

print("=== MODEL ACCURACY COMPARISON ===")
print(f"Naïve Bayes Accuracy : {nb_acc * 100:.2f}%")
print(f"Linear SVM Accuracy  : {svm_acc * 100:.2f}%")
print(f"Neural Network (MLP) : {mlp_acc * 100:.2f}%")
print("=================================\n")

print("=== SUMMARY OF PERFORMING MODELS ===")
print("1. The Neural Network (MLP) model achieved the highest overall accuracy (~85.93%), closely followed by the Linear SVM (~85.15%).")
print("2. While the SVM was highly precise in predicting cancellations (precision of 0.98), it missed many cancelled bookings (recall of only 0.41).")
print("3. The Neural Network balanced both metrics much better (precision of 0.78, recall of 0.60, resulting in a significantly higher F1-score of 0.68 for cancellations).")
print("4. Conclusion: The Neural Network is the best performing model overall for identifying booking cancellations.")

=== MODEL ACCURACY COMPARISON ===
Naïve Bayes Accuracy : 64.07%
Linear SVM Accuracy  : 85.15%
Neural Network (MLP) : 85.93%

=== SUMMARY OF PERFORMING MODELS ===
1. The Neural Network (MLP) model achieved the highest overall accuracy (~85.93%), closely followed by the Linear SVM (~85.15%).
2. While the SVM was highly precise in predicting cancellations (precision of 0.98), it missed many cancelled bookings (recall of only 0.41).
3. The Neural Network balanced both metrics much better (precision of 0.78, recall of 0.60, resulting in a significantly higher F1-score of 0.68 for cancellations).
4. Conclusion: The Neural Network is the best performing model overall for identifying booking cancellations.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. Neural networks outperformed the other models when it came to overall summarized accuracy as it balanced out the factors of percision and recall for the highest F-1 score. It's interpretatilbity was a bit more challenging as the output metric was organized differently than the others to accomodate more information.

2. I would not, I would recommend the SVM since it would likely be less expensive to run long term while still maintaining a high overall accuracy just under a point below the neural networks performance. Also from an ease of understanding perspective it could be more easily shared, interpreted, and replicated by those with less experience than a neural network which can come off as initimidating to those without a background in analytics.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend implementation of SVM as it would be cost effective while still maintaining high levels of accuracy. It is hihgly precise in predicting called bookings but may have some errors in classification after the fact. Additonal data that might be included would be data from local events of weather to see how that inluences the status of the booking in certain areas to avoid generalization of hotel locations with varying geogrpahies.

2. Overall this practice helped to remind me that the most obvious metric (in this case accuracy) is not always the best metric. As an entrepreneur I need to be able to identify the numbers that matter, not just the ones that are most obviously available to me.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [6]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 335281 bytes to assignment_12_bayes_svm_neural.html
